# Notebook 02 — ETL IEEE-CIS Fraud Detection

**Salida:** `data/clean/fraud_processed.csv`

## 0. Conexión al clúster Spark

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import col, lower, trim, length, regexp_replace, when, count, isnan, row_number
from pyspark.ml.feature import Imputer, StringIndexer, StandardScaler, VectorAssembler

spark = SparkSession.builder.appName("TFM_ETL_Fraud").getOrCreate()

## 1. Carga y join de los dos CSVs

In [2]:
RAW_PATH   = "/opt/spark-data/data/raw/"
CLEAN_PATH = "/opt/spark-data/data/clean/"

df_trans = spark.read.csv(RAW_PATH + "train_transaction.csv",  header=True)

df_ident = spark.read.csv(RAW_PATH + "train_identity.csv",  header=True)

print(f"Transacciones: {df_trans.count():,} filas, {len(df_trans.columns)} columnas")
print(f"Identidad:     {df_ident.count():,} filas, {len(df_ident.columns)} columnas")

# LEFT JOIN — conservar todas las transacciones aunque no tengan datos de identidad
df = df_trans.join(df_ident, on="TransactionID", how="left")
print(f"Tras join:     {df.count():,} filas, {len(df.columns)} columnas")

Transacciones: 590,540 filas, 394 columnas
Identidad:     144,233 filas, 41 columnas
Tras join:     590,540 filas, 434 columnas


## 2. Análisis de desbalance de clases

In [3]:
print("Distribución de isFraud:")
df.groupBy("isFraud").count().show()

total = df.count()
fraudes = df.filter(col("isFraud") == 1).count()
print(f"Tasa de fraude: {fraudes/total*100:.2f}%")

Distribución de isFraud:
+-------+------+
|isFraud| count|
+-------+------+
|      0|569877|
|      1| 20663|
+-------+------+

Tasa de fraude: 3.50%


Con solo un 3.5% de fraudes (20k frente a 570k legítimas), el dataset está fuertemente desbalanceado.

## 3. Selección de columnas y gestión de nulos

In [4]:
# Columnas con >50% nulos → eliminar directamente
n_rows = df.count()
null_pct = {}

for c in df.columns:
    n_null = df.filter(col(c).isNull() | isnan(col(c))).count()
    null_pct[c] = n_null / n_rows

cols_to_drop = [c for c, pct in null_pct.items() if pct > 0.5]
print(f"Columnas con >50% nulos eliminadas: {len(cols_to_drop)}")

df = df.drop(*cols_to_drop)
print(f"Columnas restantes: {len(df.columns)}")

Columnas con >50% nulos eliminadas: 214
Columnas restantes: 220


El dataset tiene 434 columnas tras el join, pero 214 de ellas tienen más del 50% de valores ausentes. Imputar columnas con mayoría de nulls introduciría más ruido que señal. El umbral del 50% es un criterio estándar en preprocesamiento: por encima de ese porcentaje, la columna aporta menos información que ruido al modelo.

In [5]:
# Columnas features seleccionadas manualmente para el LSTM
# (transaccionales + algunas V-features relevantes)
CAT_COLS = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType"]

NUM_COLS = [
    "TransactionAmt", "card2", "card3", "card5",
    "addr1", "addr2", "dist1",
    "C1", "C2", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C13", "C14",
    "D1", "D2", "D3", "D4",
    "V12", "V13", "V14", "V15", "V16", "V17",
    "V29", "V30", "V33", "V34",
    "V70", "V71", "V72", "V73", "V74",
    "V75", "V76", "V78", "V79", "V80", "V81", "V82", "V83"
]

ID_COLS  = ["TransactionID", "TransactionDT", "card1"]  # card1 = proxy de usuario
LABEL    = "isFraud"

# Quedarse solo con columnas disponibles en el df
available = set(df.columns)
NUM_COLS  = [c for c in NUM_COLS if c in available]
CAT_COLS  = [c for c in CAT_COLS if c in available]

print(f"Features numéricas: {NUM_COLS}")
print(f"Features categóricas: {CAT_COLS}")

df_sel = df.select(ID_COLS + [LABEL] + NUM_COLS + CAT_COLS)

Features numéricas: ['TransactionAmt', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V29', 'V30', 'V33', 'V34', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83']
Features categóricas: ['ProductCD', 'card4', 'card6', 'P_emaildomain']


¿Por qué estas columnas y no otras? Se hace una selección manual en lugar de usar todas las columnas por tres motivos:
- `TransactionAmt`, `card1-6`, `addr`, `C`, `D` features: son las variables transaccionales directas con mayor correlación documentada con isFraud.
- V-features seleccionadas (V12-V17, V29-V34, V70-V83): son los grupos de V-features con menor porcentaje de nulos y mayor varianza, las demás V-features tienen distribuciones muy similares entre sí.
- `card1` en `ID_COLS` y NO en `NUM_COLS`: `card1` es el proxy de identidad del usuario, no una feature de entrada al modelo.
- El filtro `if c in available` garantiza que el código no falla si alguna columna fue eliminada en el paso anterior.

## 4. Imputación de nulos

In [6]:
# Castear todas las numéricas a double
for c in NUM_COLS:
    df_sel = df_sel.withColumn(c, col(c).cast("double"))
    
# Rellenar nulos en numéricas con la mediana
imputer = Imputer(
    strategy="median",
    inputCols=NUM_COLS,
    outputCols=[c + "_imp" for c in NUM_COLS]
)

df_sel = imputer.fit(df_sel).transform(df_sel)

# Reemplazar columnas originales por las imputadas
for c in NUM_COLS:
    df_sel = df_sel.drop(c).withColumnRenamed(c + "_imp", c)

print("Imputación de numéricas completada")

Imputación de numéricas completada


Tres operaciones encadenadas:
- Cast a double: Algunas contienen `"NA"` o `""` en lugar de un número, el cast a double los convierte en null, que es exactamente lo que el Imputer necesita.
- Imputer con strategy='median': se usa la mediana en lugar de la media porque TransactionAmt y las V-features tienen distribuciones muy sesgadas a la derecha. La mediana es robusta a esos valores extremos y no introduce sesgo en el relleno.
- Reemplazar columna original: el Imputer crea columnas _imp nuevas, se eliminan las originales y se renombran para mantener el schema limpio.

## 5. Codificación de variables categóricas

In [7]:
# Rellenar nulos en categóricas antes del indexado
for c in CAT_COLS:
    df_sel = df_sel.fillna({c: "unknown"})

for c in CAT_COLS:
    indexer = StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    df_sel = indexer.fit(df_sel).transform(df_sel).drop(c) \
                    .withColumnRenamed(c + "_idx", c)

print(f"Schema final — {len(df_sel.columns)} columnas:")
df_sel.printSchema()

Schema final — 53 columnas:
root
 |-- TransactionID: string (nullable = true)
 |-- TransactionDT: string (nullable = true)
 |-- card1: string (nullable = true)
 |-- isFraud: string (nullable = true)
 |-- TransactionAmt: double (nullable = true)
 |-- card2: double (nullable = true)
 |-- card3: double (nullable = true)
 |-- card5: double (nullable = true)
 |-- addr1: double (nullable = true)
 |-- addr2: double (nullable = true)
 |-- C1: double (nullable = true)
 |-- C2: double (nullable = true)
 |-- C4: double (nullable = true)
 |-- C5: double (nullable = true)
 |-- C6: double (nullable = true)
 |-- C7: double (nullable = true)
 |-- C8: double (nullable = true)
 |-- C9: double (nullable = true)
 |-- C10: double (nullable = true)
 |-- C11: double (nullable = true)
 |-- C13: double (nullable = true)
 |-- C14: double (nullable = true)
 |-- D1: double (nullable = true)
 |-- D2: double (nullable = true)
 |-- D3: double (nullable = true)
 |-- D4: double (nullable = true)
 |-- V12: double (null

StringIndexer asigna un índice entero a cada categoría ordenado por frecuencia. Se prefiere sobre OneHotEncoder porque:
- OneHotEncoder expandiría la dimensionalidad significativamente, haciendo las secuencias del LSTM mucho más grandes sin ganancia proporcional en información.
- El LSTM puede aprender relaciones ordinales entre los índices con suficientes datos de entrenamiento.
- `handleInvalid='keep'` asigna un índice extra a valores no vistos durante el fit, evitando errores si aparecen dominios de email nuevos.
- `fillna('unknown')` previene que el StringIndexer falle con nulls en categóricas antes del indexado.

## 6. Ordenar por TransactionDT para las secuencias del LSTM

In [8]:
# Ordenar por usuario (card1) y tiempo (TransactionDT)

w = Window.partitionBy("card1").orderBy("TransactionDT")
df_sel = df_sel.withColumn("seq_rank", row_number().over(w))

print(f"Filas finales: {df_sel.count():,}")
df_sel.show(5)

Filas finales: 590,540
+-------------+-------------+-----+-------+--------------+-----+-----+-----+-----+-----+-----+-----+---+-----+----+---+---+----+---+----+-----+----+-----+-----+-----+-----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---------+-----+-----+-------------+--------+
|TransactionID|TransactionDT|card1|isFraud|TransactionAmt|card2|card3|card5|addr1|addr2|   C1|   C2| C4|   C5|  C6| C7| C8|  C9|C10| C11|  C13| C14|   D1|   D2|   D3|   D4|V12|V13|V14|V15|V16|V17|V29|V30|V33|V34|V70|V71|V72|V73|V74|V75|V76|V78|V79|V80|V81|V82|V83|ProductCD|card4|card6|P_emaildomain|seq_rank|
+-------------+-------------+-----+-------+--------------+-----+-----+-----+-----+-----+-----+-----+---+-----+----+---+---+----+---+----+-----+----+-----+-----+-----+-----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---------+-----+-----+-------------+--------+
|      3231806|      5822567|10068|      1|    

El LSTM modela el fraude como un problema de secuencias temporales: dadas las últimas 20 transacciones de un usuario, predice si la siguiente es fraudulenta. Para construir esas ventanas deslizantes, las transacciones deben estar ordenadas por tiempo dentro de cada usuario.
- `Window.partitionBy('card1')`: agrupa por usuario.
- `.orderBy('TransactionDT')`: ordena por timestamp de transacción dentro de cada grupo.
- `seq_rank`: número de orden de cada transacción dentro del historial del usuario.

## 7. Guardar el dataset procesado

In [9]:
df_sel.toPandas().to_csv(f"{CLEAN_PATH}fraud_processed.csv", index=False)

print(f"Guardado: {CLEAN_PATH}fraud_processed.csv")

print("\n" + "=" * 50)
print("RESUMEN ETL IEEE-CIS FRAUD")
print("=" * 50)
print(f"Filas tras join:        {df.count():>10,}")
print(f"Columnas originales:    {len(df_trans.columns) + len(df_ident.columns):>10}")
print(f"Columnas tras ETL:      {len(df_sel.columns):>10}")
print(f"Tasa de fraude:         {fraudes/total*100:>9.2f}%")
print("=" * 50)

spark.stop()

Guardado: /opt/spark-data/data/clean/fraud_processed.csv

RESUMEN ETL IEEE-CIS FRAUD
Filas tras join:           590,540
Columnas originales:           435
Columnas tras ETL:              54
Tasa de fraude:              3.50%
